## **Introduction**

The objective of the lab is to efficiently solve a general \(n^2 - 1\) puzzle problem, such as the Gem Puzzle or Mystic Square, using path-search algorithms. This means finding a sequence of actions to transform a random initial state into a goal state, optimizing:

- **Quality**: The number of actions in the solution.
- **Cost**: The total number of nodes evaluated during the search.
- **Efficiency**: A trade-off between quality and cost.

---

### Import Libraries and Constants

The necessary libraries are imported, and constants are defined for puzzle dimensions and randomization steps.


In [46]:
from collections import namedtuple, deque
from random import choice
from heapq import heappush, heappop
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Puzzle dimension (modifiable for other sizes)
PUZZLE_DIM = 3
RANDOMIZE_STEPS = 100_000

# Named tuple to represent an action
action = namedtuple('Action', ['pos1', 'pos2'])


### Puzzle Utility Functions
Generate Available Actions

Defines moves for the empty tile.


In [47]:
def available_actions(state: np.ndarray) -> list['Action']:
    """
    Find all valid moves for the empty tile (0).

    Args:
        state (np.ndarray): Current puzzle state.

    Returns:
        list[Action]: List of possible actions as swaps.
    """
    x, y = [int(_[0]) for _ in np.where(state == 0)]
    actions = []
    if x > 0:
        actions.append(action((x, y), (x - 1, y)))  # Move up
    if x < PUZZLE_DIM - 1:
        actions.append(action((x, y), (x + 1, y)))  # Move down
    if y > 0:
        actions.append(action((x, y), (x, y - 1)))  # Move left
    if y < PUZZLE_DIM - 1:
        actions.append(action((x, y), (x, y + 1)))  # Move right
    return actions

Execute an Action

Performs the swap for a selected action.

In [48]:
def do_action(state: np.ndarray, action: 'Action') -> np.ndarray:
    """
    Execute a move and return the new puzzle state.

    Args:
        state (np.ndarray): Current puzzle state.
        action (Action): Action to perform.

    Returns:
        np.ndarray: New puzzle state after the move.
    """
    new_state = state.copy()
    new_state[action.pos1], new_state[action.pos2] = new_state[action.pos2], new_state[action.pos1]
    return new_state


Manhattan Distance Heuristic


In [49]:
def manhattan_distance(state: np.ndarray) -> int:
    """
    Compute the Manhattan distance for the puzzle state.

    Args:
        state (np.ndarray): Current puzzle state.

    Returns:
        int: Total Manhattan distance of the state.
    """
    distance = 0
    for x in range(PUZZLE_DIM):
        for y in range(PUZZLE_DIM):
            value = state[x, y]
            if value != 0:  # Ignore the empty tile
                target_x, target_y = divmod(value - 1, PUZZLE_DIM)
                distance += abs(x - target_x) + abs(y - target_y)
    return distance


### Randomize Puzzle State
Generates a randomized puzzle state by performing a number of random moves.

In [50]:
def randomize_puzzle() -> np.ndarray:
    """
    Generate a randomized puzzle state by performing random moves.

    Returns:
        np.ndarray: Randomized puzzle state.
    """
    state = np.array([i for i in range(1, PUZZLE_DIM**2)] + [0]).reshape((PUZZLE_DIM, PUZZLE_DIM))
    for _ in tqdm(range(RANDOMIZE_STEPS), desc='Randomizing'):
        state = do_action(state, choice(available_actions(state)))
    return state


### A Search Algorithm*
Uses Manhattan distance as the heuristic to guide the search.

In [51]:
def a_star(initial_state: np.ndarray):
    """
    Perform A* search to solve the puzzle.

    Args:
        initial_state (np.ndarray): Randomized puzzle state.

    Returns:
        list or None: Path of states leading to the solution or None if no solution exists.
    """
    goal = np.array([i for i in range(1, PUZZLE_DIM**2)] + [0]).reshape((PUZZLE_DIM, PUZZLE_DIM))
    open_set = []
    heappush(open_set, (0, initial_state.tobytes(), 0))  # (f-score, state, g-score)
    came_from = {}
    g_score = {initial_state.tobytes(): 0}
    f_score = {initial_state.tobytes(): manhattan_distance(initial_state)}

    while open_set:
        _, current_bytes, _ = heappop(open_set)
        current = np.frombuffer(current_bytes, dtype=int).reshape((PUZZLE_DIM, PUZZLE_DIM))

        if np.array_equal(current, goal):
            return reconstruct_path(came_from, current_bytes)

        for action in available_actions(current):
            neighbor = do_action(current, action)
            neighbor_bytes = neighbor.tobytes()
            tentative_g = g_score[current_bytes] + 1

            if neighbor_bytes not in g_score or tentative_g < g_score[neighbor_bytes]:
                g_score[neighbor_bytes] = tentative_g
                f_score[neighbor_bytes] = tentative_g + manhattan_distance(neighbor)
                heappush(open_set, (f_score[neighbor_bytes], neighbor_bytes, tentative_g))
                came_from[neighbor_bytes] = current_bytes
    return None


### Reconstruct Path
Reconstructs the solution path from the start to the goal state.

In [52]:
def reconstruct_path(came_from, current):
    """
    Reconstruct the solution path.

    Args:
        came_from (dict): Map of each state to its parent state.
        current: The goal state as bytes.

    Returns:
        list: Sequence of states from start to goal.
    """
    path = []
    while current in came_from:
        path.append(current)
        current = came_from[current]
    path.reverse()
    return path


In [53]:
def evaluate_algorithm(algorithm, initial_state, puzzle_dim):
    """
    Evaluate the performance of a path-search algorithm for the n^2 - 1 puzzle.

    Args:
        algorithm (function): The algorithm to evaluate (e.g., A*, BFS).
        initial_state (np.ndarray): The randomized initial state of the puzzle.
        puzzle_dim (int): The dimension of the puzzle (e.g., 3 for a 3x3 puzzle).

    Returns:
        dict: A dictionary containing quality, cost, efficiency, and algorithm name.
    """
    global nodes_evaluated  # Keep track of nodes evaluated globally
    nodes_evaluated = 0

    # Wrapper to track nodes evaluated during search
    def count_nodes_decorator(func):
        def wrapper(*args, **kwargs):
            global nodes_evaluated
            nodes_evaluated += 1
            return func(*args, **kwargs)
        return wrapper

    # Replace the original `available_actions` and `do_action` with wrapped versions
    original_available_actions = available_actions
    original_do_action = do_action

    globals()['available_actions'] = count_nodes_decorator(original_available_actions)
    globals()['do_action'] = count_nodes_decorator(original_do_action)

    # Run the algorithm
    solution_path = algorithm(initial_state)

    # Restore the original functions
    globals()['available_actions'] = original_available_actions
    globals()['do_action'] = original_do_action

    # Compute metrics
    quality = len(solution_path) - 1 if solution_path else None
    cost = nodes_evaluated
    efficiency = 1000*quality / cost if quality and cost else None

    return {
        "algorithm": algorithm.__name__,
        "puzzle_dim": puzzle_dim,
        "quality": quality,
        "cost": cost,
        "efficiency": efficiency
    }


In [54]:
def print_and_save_results(results, output_file="algorithm_comparison.csv"):
    """
    Print the results and save them to a CSV file for visualization.

    Args:
        results (list[dict]): List of dictionaries containing evaluation metrics.
        output_file (str): File path to save the results.
    """
    import csv

    # Print results
    print(f"{'Algorithm':<20} {'Puzzle Size':<12} {'Quality':<10} {'Cost':<10} {'Efficiency':<12}")
    print("=" * 70)
    for result in results:
        print(f"{result['algorithm']:<20} {result['puzzle_dim']}x{result['puzzle_dim']:<10} "
              f"{result['quality']:<10} {result['cost']:<10} {result['efficiency']:<12.4f}")

    # Save results to CSV
    with open(output_file, mode="w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["algorithm", "puzzle_dim", "quality", "cost", "efficiency"])
        writer.writeheader()
        writer.writerows(results)

    print(f"Results saved to {output_file}")


In [55]:
def plot_results_from_csv(file_path):
    """
    Read results from a CSV file and generate comparative plots for quality, cost, and efficiency.

    Args:
        file_path (str): Path to the CSV file containing the results.
    """
    # Read the CSV file into a DataFrame
    results = pd.read_csv(file_path)

    # Extract unique algorithms and puzzle dimensions
    algorithms = results['algorithm'].unique()
    puzzle_sizes = sorted(results['puzzle_dim'].unique())

    # Initialize plots for quality, cost, and efficiency
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
    axes[0].set_title("Quality vs Puzzle Size")
    axes[1].set_title("Cost vs Puzzle Size")
    axes[2].set_title("Efficiency vs Puzzle Size")

    for algorithm in algorithms:
        # Filter data for the current algorithm
        algo_results = results[results['algorithm'] == algorithm]

        # Plot Quality
        axes[0].plot(
            algo_results['puzzle_dim'],
            algo_results['quality'],
            marker='o',
            label=algorithm
        )

        # Plot Cost
        axes[1].plot(
            algo_results['puzzle_dim'],
            algo_results['cost'],
            marker='o',
            label=algorithm
        )

        # Plot Efficiency
        axes[2].plot(
            algo_results['puzzle_dim'],
            algo_results['efficiency'],
            marker='o',
            label=algorithm
        )

    # Set labels and legends
    for ax in axes:
        ax.set_xlabel("Puzzle Size (n x n)")
        ax.set_xticks(puzzle_sizes)
        ax.set_xticklabels([f"{size}x{size}" for size in puzzle_sizes])
        ax.legend()
        ax.grid()

    axes[0].set_ylabel("Quality (Number of Actions)")
    axes[1].set_ylabel("Cost (Nodes Evaluated)")
    axes[2].set_ylabel("Efficiency (Quality / Cost)")

    # Show the plots
    plt.tight_layout()
    plt.show()


### Main Execution
Randomizes the initial state, solves the puzzle using A*, and displays the solution.

In [ ]:
if __name__ == "__main__":
    # Define puzzles of increasing sizes
    puzzle_sizes = [4]
    results = []
    print("\nInitial state:")
    print_state(state)
    for size in puzzle_sizes:
        PUZZLE_DIM = size
        initial_state = randomize_puzzle()

        # Evaluate A* algorithm
        result_a_star = evaluate_algorithm(a_star, initial_state, size)
        results.append(result_a_star)

    # Print and save results
    print_and_save_results(results)
    # plot_results_from_csv("algorithm_comparison.csv")


Randomizing:   0%|          | 0/100000 [00:00<?, ?it/s]